# Demo do Classificador — NASA Kepler Objects of Interest

Este notebook é uma **demonstração interativa** do modelo treinado para classificar objetos da missão Kepler em:

- **CANDIDATE** — Candidato a exoplaneta real
- **FALSE POSITIVE** — Falso positivo (ruído, artefato instrumental ou fenômeno estelar)


## ️ 1. Importações e Configurações

In [1]:
import pandas as pd # Manipulação de dados tabulares
import numpy as np # Operações numéricas e arrays
import warnings # Supressão de avisos não críticos

from sklearn.model_selection import train_test_split # Divisão estratificada dos dados
from sklearn.preprocessing import StandardScaler, LabelEncoder # Escalonamento e encoding
from sklearn.neural_network import MLPClassifier # Rede Neural MLP (modelo final)
from sklearn.metrics import accuracy_score # Métrica de acurácia

warnings.filterwarnings('ignore') # Oculta warnings irrelevantes para a demo
pd.set_option('display.max_columns', None) # Exibe todas as colunas no Pandas

print(" Bibliotecas carregadas com sucesso!") # Confirmação de carregamento

 Bibliotecas carregadas com sucesso!


## 2. Carregamento dos Dados

Os dados são baixados diretamente do **NASA Exoplanet Archive** via protocolo TAP — a mesma fonte usada em todo o projeto.

In [2]:
url = 'https://exoplanetarchive.ipac.caltech.edu/TAP/sync?query=select+*+from+cumulative&format=csv' # URL oficial TAP do Exoplanet Archive
df_raw = pd.read_csv(url) # Baixa o dataset KOI diretamente via TAP (URL)

print(f" Dataset carregado: {df_raw.shape[0]:,} registros e {df_raw.shape[1]} colunas") # Exibe dimensões do dataset

 Dataset carregado: 9,564 registros e 153 colunas


## 3. Pré-processamento (Pipeline Completo)

Replicamos aqui o mesmo pipeline do notebook `02_preprocessamento_modelagem.ipynb` para garantir consistência total.

In [3]:
# ── Seleção do alvo e remoção de colunas irrelevantes ──────────────────────
target = 'koi_pdisposition' # Define a coluna alvo da classificação

cols_to_drop = [ # Lista de colunas irrelevantes a remover (IDs e metadados)
 'kepid', 'kepoi_name', 'kepler_name', 'koi_disposition',
 'koi_tce_delivname', 'koi_fittype', 'ra_str', 'dec_str'
]

null_threshold = 0.5 * len(df_raw) # Limite de 50% de nulos para remoção de coluna
high_null_cols = df_raw.columns[df_raw.isnull().sum() > null_threshold].tolist() # Identifica colunas com mais de 50% de nulos
total_drop = list(set(cols_to_drop + high_null_cols)) # Combina lista de colunas para remoção
df = df_raw.drop(columns=total_drop) # Remove colunas irrelevantes do dataframe

# ── Encoding e imputação ───────────────────────────────────────────────────
le = LabelEncoder() # Cria codificador de rótulos categóricos
df[target] = le.fit_transform(df[target]) # Aprende e transforma rótulos CANDIDATE/FALSE POSITIVE → 0/1

df_numeric = df.select_dtypes(include=[np.number]) # Seleciona apenas colunas numéricas
df_final = df_numeric.fillna(df_numeric.median()) # Preenche nulos com a mediana de cada coluna

# ── Split 70/15/15 ─────────────────────────────────────────────────────────
X = df_final.drop(columns=[target]) # Separa features (entradas) removendo coluna alvo
y = df_final[target] # Extrai a coluna alvo como variável dependente

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y) # Primeiro split: 70% treino, 30% restante
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp) # Segundo split: divide 30% em 15% validação e 15% teste

# ── Escalonamento ──────────────────────────────────────────────────────────
scaler = StandardScaler() # Cria o padronizador (média=0, desvio=1)
X_train_scaled = scaler.fit_transform(X_train) # Aprende e aplica escalonamento no treino
X_val_scaled = scaler.transform(X_val) # Escala validação com parâmetros do treino
X_test_scaled = scaler.transform(X_test) # Escala teste com parâmetros do treino

print(f" Pipeline concluído!") # Confirmação de conclusão
print(f" Treino: {X_train.shape[0]:,} amostras") # Exibe dimensão do conjunto de treino
print(f" Validação: {X_val.shape[0]:,} amostras") # Exibe dimensão do conjunto de validação
print(f" Teste: {X_test.shape[0]:,} amostras (nunca visto pelo modelo)") # Exibe dimensão do conjunto de teste

 Pipeline concluído!
 Treino: 6,694 amostras
 Validação: 1,435 amostras
 Teste: 1,435 amostras (nunca visto pelo modelo)


## 4. Treinamento do Modelo Final

Modelo escolhido: **Experimento C** — MLP com 1 camada oculta (100 neurônios) e regularização L2 (`alpha=0.01`).

> Esta foi a configuração que obteve a **maior acurácia de validação (98,40%)** dentre todos os experimentos realizados.

In [4]:
modelo_final = MLPClassifier( # Cria MLP Exp C: regularização L2 (alpha=0.01)
 hidden_layer_sizes=(100,), # 1 camada oculta com 100 neurônios
 alpha=0.01, # Regularização L2 para evitar overfitting
 max_iter=500, # Máximo de 500 épocas de treinamento
 random_state=42, # Semente para reprodutibilidade
 early_stopping=True # Para o treino se não houver melhora
)

print("Treinando o modelo... (pode levar alguns segundos)") # Aviso de início do treino
modelo_final.fit(X_train_scaled, y_train) # Treina a rede neural com os dados de treino

acc = accuracy_score(y_val, modelo_final.predict(X_val_scaled)) # Calcula acurácia do modelo na validação
print(f"\n Modelo treinado!") # Confirmação de treinamento concluído
print(f" Acurácia na Validação: {acc:.2%}") # Exibe resultado formatado como porcentagem

Treinando o modelo... (pode levar alguns segundos)

 Modelo treinado!
 Acurácia na Validação: 98.40%


## 5. Demo ao Vivo — Predições em Dados Reais

Abaixo, o modelo classifica **10 objetos aleatórios** do conjunto de **teste** — dados que ele **nunca viu** durante o treinamento.

A tabela mostra:
- **Índice** do objeto no dataset original
- **Classe Real** (o gabarito — o que a NASA registrou)
- **Predição do Modelo** (o que a rede neural inferiu)
- **Resultado** Acerto ou Erro

In [5]:
np.random.seed(7) # Semente fixa para reprodutibilidade dos exemplos da demo
indices_demo = np.random.choice(len(X_test), size=10, replace=False) # Seleciona 10 índices aleatórios do teste

X_demo = X_test_scaled[indices_demo] # Extrai as amostras selecionadas (já escalonadas)
y_real = y_test.iloc[indices_demo].values # Rótulos reais correspondentes
y_pred = modelo_final.predict(X_demo) # Gera predições do modelo final para as amostras
y_proba = modelo_final.predict_proba(X_demo) # Obtém probabilidades de pertencer a cada classe

# ── Construção da tabela de resultados ────────────────────────────────────
rows = [] # Lista para armazenar cada linha da tabela de resultados
for i, (real, pred, proba) in enumerate(zip(y_real, y_pred, y_proba)): # Itera sobre cada exemplo
 classe_real = le.inverse_transform([real])[0] # Converte label numérico → nome da classe real
 classe_pred = le.inverse_transform([pred])[0] # Converte label numérico → nome da classe predita
 confianca = max(proba) * 100 # Extrai a maior probabilidade como confiança do modelo
 acerto = ' Correto' if real == pred else ' Erro' # Define ícone de acerto ou erro
 rows.append({ # Adiciona linha à tabela de resultados
 'Classe Real': classe_real,
 'Predição do Modelo': classe_pred,
 'Confiança (%)': f"{confianca:.1f}%",
 'Resultado': acerto
 })

df_demo = pd.DataFrame(rows) # Cria DataFrame com os resultados da demo
df_demo.index = df_demo.index + 1 # Começa índice em 1 para exibição mais legível

print("\n PREDIÇÕES DO MODELO — 10 Objetos Aleatórios do Conjunto de Teste\n") # Cabeçalho da demo
display(df_demo) # Exibe tabela formatada no Jupyter

n_acertos = sum(r == p for r, p in zip(y_real, y_pred)) # Conta acertos dentre os 10 exemplos
print(f"\n Resultado da Demo: {n_acertos}/10 acertos ({n_acertos*10}% nesta amostra)") # Exibe placar da demo


 PREDIÇÕES DO MODELO — 10 Objetos Aleatórios do Conjunto de Teste



,Classe Real,Predição do Modelo,Confiança (%),Resultado
1,CANDIDATE,CANDIDATE,100.0%,Correto
2,CANDIDATE,CANDIDATE,100.0%,Correto
3,FALSE POSITIVE,FALSE POSITIVE,100.0%,Correto
4,CANDIDATE,CANDIDATE,100.0%,Correto
5,CANDIDATE,CANDIDATE,100.0%,Correto
6,FALSE POSITIVE,FALSE POSITIVE,100.0%,Correto
7,CANDIDATE,CANDIDATE,99.9%,Correto
8,CANDIDATE,CANDIDATE,100.0%,Correto
9,FALSE POSITIVE,FALSE POSITIVE,100.0%,Correto
10,FALSE POSITIVE,FALSE POSITIVE,100.0%,Correto



 Resultado da Demo: 10/10 acertos (100% nesta amostra)


## 6. Teste Manual — Insira os Dados de um Objeto

Você pode testar o modelo com **qualquer linha do dataset original**. Basta informar o índice do objeto.

In [16]:
# ────────────────────────────────────────────────────────────────────────────
# Altere este índice para testar qualquer objeto do conjunto de teste!
INDICE_DO_OBJETO = 950 # Índice dentro do X_test (0 = primeiro objeto do teste)
# ────────────────────────────────────────────────────────────────────────────

objeto = X_test_scaled[[INDICE_DO_OBJETO]] # Seleciona o objeto pelo índice e mantém 2D
real = y_test.iloc[INDICE_DO_OBJETO] # Obtém o rótulo real correspondente

pred = modelo_final.predict(objeto)[0] # Gera predição do modelo para o objeto selecionado
proba = modelo_final.predict_proba(objeto)[0] # Obtém probabilidades de cada classe

classe_real = le.inverse_transform([real])[0] # Decodifica rótulo real para texto
classe_pred = le.inverse_transform([pred])[0] # Decodifica predição para texto

print("\n" + "=" * 45) # Linha separadora decorativa
print(f" Objeto de Teste #{INDICE_DO_OBJETO}") # Exibe índice do objeto avaliado
print("=" * 45) # Linha separadora decorativa
print(f" Classe Real (NASA): {classe_real}") # Exibe a classe registrada pela NASA
print(f" Predição do Modelo: {classe_pred}") # Exibe o que a rede neural classificou
print(f" Confiança (CANDIDATE): {proba[0]*100:.1f}%") # Probabilidade de ser CANDIDATE
print(f" Confiança (FALSE POS): {proba[1]*100:.1f}%") # Probabilidade de ser FALSE POSITIVE
print("=" * 45) # Linha separadora decorativa
print(f" Resultado: {' ACERTO!' if real == pred else ' ERRO!'}") # Resultado final
print("=" * 45) # Linha separadora decorativa


 Objeto de Teste #950
 Classe Real (NASA): CANDIDATE
 Predição do Modelo: CANDIDATE
 Confiança (CANDIDATE): 85.6%
 Confiança (FALSE POS): 14.4%
 Resultado:  ACERTO!


---

## Resumo do Projeto

| Etapa | Resultado |
|---|---|
| Dataset | 9.564 registros × 153 colunas (NASA KOI Cumulative) |
| Features após limpeza | 108 atributos físicos e astronômicos |
| Divisão | 70% treino / 15% validação / 15% teste |
| Baseline (Regressão Logística) | **93,45%** de acurácia |
| MLP v1 (100 neurônios, padrão) | **98,33%** de acurácia |
| **Modelo Final (Exp C — Alpha=0.01)** | **98,40%** de acurácia |

<br>

---

<br>


> **Biblioteca:** Scikit-Learn `MLPClassifier` · **Ativação:** ReLU · **Otimizador:** Adam · **Regularização:** L2 (`alpha=0.01`)